# Tutorial 3: Quantitative Backtesting

Run backtests with built-in or custom strategies using FinRobot's BackTrader adapter.

**What you'll learn:**
- How to configure and run a backtest
- How to interpret BacktestResult (Sharpe, drawdown, win rate)
- How to use the LLM strategy selection agent

**Requirements:**
- `pip install 'finrobot[backtest]'` (installs backtrader)

## Basic Backtest with SMA Crossover

In [ ]:
from finrobot import FinRobot
from finrobot.engine.backtest.engine import BacktestConfig

agent = FinRobot(model="deepseek:deepseek-chat")

# Configure a simple SMA crossover backtest
config = BacktestConfig(
    ticker="AAPL",
    start_date="2023-01-01",
    end_date="2024-12-31",
    strategy="sma_crossover",
    strategy_params={"fast": 10, "slow": 30},
    initial_cash=100_000.0,
)

result = await agent.abacktest(config)
print(result.format_summary())

## Analyze Results

In [ ]:
print(f"Total Return:  {result.total_return:+.2%}")
print(f"Sharpe Ratio:  {result.sharpe_ratio:.3f}" if result.sharpe_ratio else "Sharpe: N/A")
print(f"Max Drawdown:  {result.max_drawdown:.2%}" if result.max_drawdown else "Drawdown: N/A")
print(f"Win Rate:      {result.win_rate:.1%}" if result.win_rate else "Win Rate: N/A")

## Compare Different Parameters

Try different SMA periods to see how they affect performance.

In [ ]:
param_sets = [
    {"fast": 5, "slow": 20},
    {"fast": 10, "slow": 30},
    {"fast": 20, "slow": 50},
]

for params in param_sets:
    cfg = BacktestConfig(
        ticker="AAPL",
        start_date="2023-01-01",
        end_date="2024-12-31",
        strategy="sma_crossover",
        strategy_params=params,
    )
    r = await agent.abacktest(cfg)
    sharpe = f"{r.sharpe_ratio:.2f}" if r.sharpe_ratio else "N/A"
    print(
        f"SMA({params['fast']},{params['slow']}): Return={r.total_return:+.2%}  Sharpe={sharpe}  Trades={r.total_trades}"
    )

## LLM Strategy Selection (Optional)

Let the LLM choose strategy parameters and iteratively optimize.

In [ ]:
from finrobot.engine.backtest.strategy_agent import run_strategy_selection
from finrobot.config import get_settings

settings = get_settings(model_name="deepseek:deepseek-chat")
best_result = await run_strategy_selection(
    settings=settings,
    ticker="AAPL",
    start_date="2023-01-01",
    end_date="2024-12-31",
)
print("\nBest result from LLM strategy selection:")
print(best_result.format_summary())

In [ ]:
await agent.close()